In [6]:
import os
import json
import math
import pandas as pd

# -------------------------
# CONFIG
# -------------------------

# If you export to CSV, use this:
INPUT_PATH = "Experiment_Order_Expanded.xlsx"
READ_AS_EXCEL = False  # set True if you want to read the original .xlsx

# If you want to read the .xlsx directly instead, set:
# INPUT_PATH = "Experiment_Order_Expanded.xlsx"
# READ_AS_EXCEL = True

OUTPUT_DIR = "participants_json"
COMBINED_JSON = "participants_all.json"

# -------------------------
# LOAD FILE
# -------------------------

if READ_AS_EXCEL:
    df = pd.read_excel(INPUT_PATH)
else:
# If the input file is an Excel workbook, prefer reading it as Excel
    if not READ_AS_EXCEL and INPUT_PATH.lower().endswith(('.xls', '.xlsx')):
        print(f"Input file '{INPUT_PATH}' looks like Excel — reading with pd.read_excel().")
        READ_AS_EXCEL = True

if READ_AS_EXCEL:
    df = pd.read_excel(INPUT_PATH)
else:
# Try several encodings because the CSV may not be UTF-8
# (Windows Hebrew files often use 'cp1255').
    encodings_to_try = ["utf-8", "utf-8-sig", "cp1255", "cp1252", "iso-8859-1", "latin1"]
    last_exc = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(INPUT_PATH, encoding=enc)
            print(f"Read CSV using encoding: {enc}")
            break
        except Exception as e:
            last_exc = e
    else:
# All attempts failed; re-raise the last exception to show the error
        raise last_exc

# -------------------------
# HELPERS
# -------------------------

def clean(value):
    """Convert NaN to None, leave other values as-is."""
    if isinstance(value, float) and math.isnan(value):
        return None
    return value

participants = {}

# -------------------------
# MAIN LOOP: PER PARTICIPANT
# -------------------------

for _, row in df.iterrows():
    pid = row["Participant_ID"]

    participant = {
        "participant_id": pid,
        "condition_order_text": clean(row.get("סדר_תנאים")),
        "practice": [],
        "conditions": []
    }

    # -------------------------
    # PRACTICE TRIALS
    # -------------------------
    # Columns pattern:
    #   תרגול_R1_Scenario_ID
    #   תרגול_R1_קושי
    #   תרגול_R1_תשובה_נכונה_לתרחיש
    #   תרגול_R1_המלצת_מערכת
    #   תרגול_R1_Q1_נכון / Q2_נכון / Q3_נכון

    for r in [1, 2, 3]:
        base = f"תרגול_R{r}_"
        scenario_id = row.get(base + "Scenario_ID")

        if pd.isna(scenario_id):
            # No practice trial in this slot
            continue

        trial = {
            "slot": r,
            "scenario_id": clean(scenario_id),
            "difficulty": clean(row.get(base + "קושי")),
            "correct_route": clean(row.get(base + "תשובה_נכונה_לתרחיש")),
            "ai_recommended_route": clean(row.get(base + "המלצת_מערכת")),
            "correct_answers": {
                "Q1": clean(row.get(base + "Q1_נכון")),
                "Q2": clean(row.get(base + "Q2_נכון")),
                "Q3": clean(row.get(base + "Q3_נכון")),
            }
        }
        participant["practice"].append(trial)

    # -------------------------
    # CONDITIONS (1, 2, 3)
    # -------------------------
    # For each condition i:
    #   תנאי_i_ויזואליזציה
    #   תנאי_i_סדר_מודלים
    #   תנאי_i_M1_סוג_מודל, תנאי_i_M2_סוג_מודל
    #   תנאי_i_M1_R1_Scenario_ID, ... up to R5
    #   תנאי_i_M1_R1_קושי / ... / Q3_נכון
    #   (same for M2)

    for cond_index in [1, 2, 3]:
        prefix = f"תנאי_{cond_index}_"

        cond = {
            "index": cond_index,
            "visualization": clean(row.get(prefix + "ויזואליזציה")),
            "model_order_text": clean(row.get(prefix + "סדר_מודלים")),
            "models": []
        }

        # Models M1 and M2
        for m_tag in ["M1", "M2"]:
            model_type_col = prefix + f"{m_tag}_סוג_מודל"
            if model_type_col not in df.columns:
                # Should not happen, but keep it robust
                model_type = None
            else:
                model_type = clean(row.get(model_type_col))

            model = {
                "tag": m_tag,            # "M1" or "M2"
                "model_type": model_type,  # "OPT" / "SUB" (as in the sheet)
                "trials": []
            }

            # up to 5 repetitions per model
            for r in [1, 2, 3, 4, 5]:
                base = f"{prefix}{m_tag}_R{r}_"
                scenario_col = base + "Scenario_ID"

                if scenario_col not in df.columns:
                    # No such column (just in case)
                    continue

                scenario_id = row.get(scenario_col)

                # If empty / NaN → no trial in this slot
                if pd.isna(scenario_id):
                    continue

                trial = {
                    "slot": r,
                    "scenario_id": clean(scenario_id),
                    "difficulty": clean(row.get(base + "קושי")),
                    "correct_route": clean(row.get(base + "תשובה_נכונה_לתרחיש")),
                    "ai_recommended_route": clean(row.get(base + "המלצת_מערכת")),
                    "correct_answers": {
                        "Q1": clean(row.get(base + "Q1_נכון")),
                        "Q2": clean(row.get(base + "Q2_נכון")),
                        "Q3": clean(row.get(base + "Q3_נכון")),
                    }
                }

                model["trials"].append(trial)

            cond["models"].append(model)

        participant["conditions"].append(cond)

    participants[pid] = participant

# -------------------------
# WRITE JSON FILES
# -------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

# One file per participant
for pid, pdata in participants.items():
    out_path = os.path.join(OUTPUT_DIR, f"{pid}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(pdata, f, ensure_ascii=False, indent=2)

# Combined file with all participants
with open(COMBINED_JSON, "w", encoding="utf-8") as f:
    json.dump(list(participants.values()), f, ensure_ascii=False, indent=2)

print(f"Written {len(participants)} participant JSON files to '{OUTPUT_DIR}'")
print(f"Combined JSON written to '{COMBINED_JSON}'")


Input file 'Experiment_Order_Expanded.xlsx' looks like Excel — reading with pd.read_excel().
Written 30 participant JSON files to 'participants_json'
Combined JSON written to 'participants_all.json'
Written 30 participant JSON files to 'participants_json'
Combined JSON written to 'participants_all.json'
